In [29]:
# Testing --> Tesing AND validation
# Onehot-Encode
# DataLoader support

In [91]:
import numpy as np
import torch
from torch.utils.data import Dataset
from torchvision.io import read_image
from torchvision.transforms import v2
from sklearn.preprocessing import LabelEncoder
import os
import glob
from tqdm.auto import tqdm

class LoadBRISC():
    def __init__(self):
        self.cls_root = './brisc2025/classification_task'
        self.seg_root = './brisc2025/segmentation_task'
    
    def load(self,classes:str='all',planes:str='all',label_type:str='class',encoding_type='integer',train_transform=None,test_transform=None):
        '''
        Loads BRISC 2025 dataset into a PyTorch Dataset.

        Args:
            classes (str, default='all'): Classes to pick.
                Supported values are:
                    - 'all' : Picks all classes
                    - 'tumor' : Picks all classes except 'no_tumor' 
            planes (str: default='all') : Planes to pick
                Supported values are:
                    - 'all' : Picks all planes
                    - 'ax' : Axial plane
                    - 'sa' : Sagital plane
                    - 'co' : Coronal plane
            label_type (str, default='class'): Label return type
                Supported values are:
                    - 'class' : Returns tumor types as labels
                    - 'plane' : Returns plane types as labels
                    - 'both' : Returns tumor and plane types as labels
            encoding_type (str, default='integer'): Label encoding type
                Supported values are:
                    - 'onehot' : Returns labels in onehot encoding
                    - 'integer' : Returns labels in integer labels
            train_transform (torchvision.transforms): Image transformation for the training dataset
            test_transform (torchvision.transforms): Image transformation for the testing dataset

        Returns:
            PyTorch Dataset object of selected configuration.
        '''
        if classes not in ['all','tumor']:
            raise ValueError('Unknown value for "classes" arguement! Use "all" or "tumor".')
        if planes not in ['all','ax','sa','co']:
            raise ValueError('Unknown value for "planes" arguement! Use "all", "ax", "sa", or "co".')
        if label_type not in ['class','plane','both']:
            raise ValueError('Unknown value for "label_type" arguement! Use "class", "plane", or "both".')
        if encoding_type not in ['onehot','integer']:
            raise ValueError('Unknown value for "encoding_type" arguement! Use "integer", or "onehot".')

        self.classes_dict = {
            'all':['gl','me','no','pi'],
            'tumor':['gl','me','pi']
        }
        self.planes_dict = {
            'all':['ax','sa','co'],
            'ax':['ax'],
            'sa':['sa'],
            'co':['co']
        }
        self.classes = self.classes_dict[classes]
        self.planes = self.planes_dict[planes]
        self.encoding_type = encoding_type

        # Get image paths and labels
        train_img_paths = []
        train_mask_paths = []
        train_labels = []
        test_img_paths = []
        test_mask_paths = []
        test_labels = []
        for cls in self.classes:
            for pln in self.planes:
                labels = f'{cls}' if label_type == 'class' else f'{pln}' if label_type == 'plane' else f'{cls} {pln}'
                if cls == 'no':
                    train_paths = glob.glob(os.path.join(self.cls_root,'train','no_tumor',f'*{pln}*'))
                    test_paths = glob.glob(os.path.join(self.cls_root,'test','no_tumor',f'*{pln}*'))

                    train_img_paths.extend(train_paths)
                    test_img_paths.extend(test_paths)

                    train_mask_paths.extend([None for i in range(len(train_paths))])
                    test_mask_paths.extend([None for i in range(len(test_paths))])
                else:
                    train_paths = glob.glob(os.path.join(self.seg_root,'train','images',f'*{cls}_{pln}*'))
                    test_paths = glob.glob(os.path.join(self.seg_root,'test','images',f'*{cls}_{pln}*'))

                    train_img_paths.extend(train_paths)
                    test_img_paths.extend(test_paths)

                    train_mask_paths.extend(glob.glob(os.path.join(self.seg_root,'train','masks',f'*{cls}_{pln}*')))
                    test_mask_paths.extend(glob.glob(os.path.join(self.seg_root,'test','masks',f'*{cls}_{pln}*')))
                train_labels.extend([labels for i in range(len(train_paths))])
                test_labels.extend([labels for i in range(len(test_paths))])

        # Get images and masks
        train_imgs,train_masks = self.__import_image(train_img_paths,train_mask_paths)
        test_imgs,test_masks = self.__import_image(test_img_paths,test_mask_paths)
        train_labels = np.array(train_labels)
        test_labels = np.array(test_labels)

        # Encode labels
        train_labels = self.__encode_label(train_labels)
        test_labels = self.__encode_label(test_labels)

        # Put images and masks into a Dataset object
        self.train_ds = DatasetClass(train_imgs,train_masks,train_labels,transform=train_transform)
        self.test_ds = DatasetClass(test_imgs,test_masks,test_labels,transform=test_transform)
        return self.train_ds,self.test_ds

    def __encode_label(self,labels:np.ndarray):
        '''
        Encodes labels into integer or onehot encoding
        Args:
            labels (np.ndarray): Array of labels to encode
        Returns:
            Array of encoded labels
        '''
        encoder = LabelEncoder()
        labels_enc = encoder.fit_transform(labels)
        self.classes = encoder.classes_
        if self.encoding_type == 'onehot':
            labels_enc = np.eye(len(self.classes),dtype=int)[labels_enc]
        return labels_enc

    def __import_image(self,img_paths,mask_paths):
        '''
        Imports images and masks from given list of paths
        Args:
            img_paths (list): List of path to images
            mask_paths (list) : List of path to masks
        
        Returns:
            PyTorch tensor of the images and masks in one batch
        '''
        imgs,masks = [],[]
        for i in tqdm(range(len(img_paths))):
            img = read_image(img_paths[i])
            img = v2.functional.resize(img,(224,224))
            img = v2.Grayscale(num_output_channels=1)(img)
            if mask_paths[i]:
                mask = read_image(mask_paths[i])
                mask = v2.functional.resize(mask,(224,224))
            else:
                mask = torch.zeros_like(img)
 
            imgs.append(img)
            masks.append(mask)
        imgs = torch.cat(imgs).to(torch.float32).unsqueeze(1)
        masks = torch.cat(masks).to(torch.float32).unsqueeze(1)
        return imgs,masks

class DatasetClass(Dataset):
    def __init__(self,imgs,masks,labels,transform=None):
        self.imgs = imgs
        self.masks = masks
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return self.imgs.shape[0]

    def __getitem__(self, index):
        img = self.imgs[index]
        mask = self.masks[index]
        label = self.labels[index]

        if self.transform:
            img = self.transform(img)

        return img,mask,label

In [90]:
loader = LoadBRISC()
train_ds,test_ds = loader.load(classes='tumor',planes='ax',encoding_type='onehot')

  0%|          | 0/1243 [00:00<?, ?it/s]

  0%|          | 0/346 [00:00<?, ?it/s]

In [63]:
from torch.utils.data import DataLoader

train_dl = DataLoader(train_ds,
                      batch_size=4,
                      shuffle=True,
                      drop_last=True)
test_dl = DataLoader(test_ds,
                     batch_size=4,
                     shuffle=False,
                     drop_last=True)

In [64]:
import torch
from torch import nn
from torchvision import models

class CAM(nn.Module):
    def __init__(self,num_class,backbone='vgg'):
        super().__init__()
        if backbone == 'vgg':
            self.pretrain = models.vgg19_bn(weights=models.VGG19_BN_Weights.IMAGENET1K_V1)
            self.backbone = self.pretrain.features
            del self.pretrain.avgpool
            del self.pretrain.classifier
        elif backbone == 'resnet':
            self.pretrain = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
            self.backbone = nn.Sequential(*list(models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).children())[:-2])
            del self.pretrain.avgpool
            del self.pretrain.fc
        elif backbone == 'effnet':
            self.pretrain = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
            self.backbone = nn.Sequential(*list(models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1).children())[:-2])
            del self.pretrain.avgpool
            del self.pretrain.classifier
        else:
            raise ValueError('Unknown backbone! Use "vgg", "resnet", or "effnet".')
        
        dummy_input = torch.zeros(1, 3, 224, 224)
        with torch.no_grad():
            dummy_output = self.backbone(dummy_input)
        fmaps_count = dummy_output.shape[1]
        
        self.classifier  = nn.Sequential(
            nn.Conv2d(fmaps_count,128,kernel_size=3,stride=1,padding=1),
            nn.BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
            nn.ReLU(),
            nn.Conv2d(128,128,kernel_size=3,stride=1,padding=1),
            nn.BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
            nn.ReLU(),
            nn.Conv2d(128,num_class,kernel_size=1,stride=1,padding=0)
        )
        self.gap = nn.AdaptiveAvgPool2d(output_size=(1,1))
    
    def forward(self,x):
        x = x.expand(-1,3,-1,-1)
        fmaps = self.backbone(x)
        fmaps_cls = self.classifier(fmaps)
        logits = self.gap(fmaps_cls).squeeze()
        preds = torch.argmax(logits,dim=1)

        self.cams = self.cam_to_mask(fmaps_cls,preds)
        return logits
    
    def cam_to_mask(self,cams,preds,threshold=0.7):
        cams = torch.relu(cams)
        cams = cams[np.arange(cams.shape[0]),preds,:,:]
        cams_min = cams.min(dim=-1)[0].min(dim=-1)[0].unsqueeze(-1).unsqueeze(-1)
        cams_max = cams.max(dim=-1)[0].max(dim=-1)[0].unsqueeze(-1).unsqueeze(-1)

        norm_cam = ((cams-cams_min)/(cams_max-cams_min+1e-8)).unsqueeze(1)
        norm_cam = torch.nn.functional.interpolate(norm_cam,(224,224))
        
        preds_masks = norm_cam >= threshold
        return preds_masks

In [85]:
import torch
from torch import nn
from tqdm.auto import tqdm
from torchmetrics.functional.segmentation import dice_score
from sklearn.metrics import accuracy_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def train_step(model,criterion,optimizer,dataloader,device=device):
    model.to(device)
    model.train()
    losses, accs, dscs = 0,0,0
    for batch,(X,M,y) in enumerate(dataloader):
        X,M,y = X.to(device),M.to(device),y.to(device)
        y_logit = model(X)
        y_mask = model.cams
        y = y.to(torch.float32)
        y_pred = torch.softmax(y_logit,1)
        loss = criterion(y_logit,y)
        losses += loss
        accs += accuracy_score(y.argmax(1).cpu(),y_pred.argmax(1).cpu().detach().numpy())
        dscs += dice_score(y_mask,M,num_classes=2).sum().item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    losses /= len(dataloader)
    losses = losses.cpu().detach().item()
    accs /= len(dataloader)
    dscs /= len(dataloader)
    return losses,accs,dscs

def val_step(model,criterion,dataloader,device=device):
    model.to(device)
    model.eval()
    losses, accs, dscs = 0,0,0
    with torch.inference_mode():
        for batch,(X,M,y) in enumerate(dataloader):
            X,M,y = X.to(device),M.to(device),y.to(device)
            y_logit = model(X)
            y_mask = model.cams
            y = y.to(torch.float32)
            y_pred = torch.softmax(y_logit,1)

            losses += criterion(y_logit,y).cpu().detach().item()
            accs += accuracy_score(y.argmax(1).cpu(),y_pred.argmax(1).cpu().detach().numpy())
            dscs += dice_score(y_mask,M,num_classes=2).sum().item()
        losses /= len(dataloader)
        accs /= len(dataloader)
        dscs /= len(dataloader)
    return losses,accs,dscs

def train_loop(model,epochs,criterion,optimizer,device,train_dataloader,val_dataloader,scheduler=None):
    metrics = {'train_loss':[],
               'train_acc':[],
               'train_dsc':[],
               'val_loss':[],
               'val_acc':[],
               'val_dsc':[]}

    for epoch in tqdm(range(epochs)):
        print(f'Epoch {epoch}:')
        train_loss,train_acc,train_dsc = train_step(model,criterion,optimizer,train_dataloader,device)
        val_loss,val_acc,val_dsc = val_step(model,criterion,val_dataloader,device)

        if scheduler:
            scheduler.step(val_loss)
            
        metrics['train_loss'].append(train_loss)
        metrics['train_acc'].append(train_acc)
        metrics['train_dsc'].append(train_dsc)
        metrics['val_loss'].append(val_loss)
        metrics['val_acc'].append(val_acc)
        metrics['val_dsc'].append(val_dsc)
        print(f'Train Loss: {train_loss:.3f} | Train Acc: {train_acc:.2f} | Train DSC: {train_dsc:.2f}| Val Loss: {val_loss:.3f} | Val Acc: {val_acc:.2f} | Val DSC: {val_dsc:.2f}')
    return metrics

In [87]:
model = CAM(3,backbone='effnet')
CRITERION = nn.CrossEntropyLoss()
OPTIMIZER = torch.optim.SGD(model.parameters(),lr=0.1)
history = train_loop(model,2,CRITERION,OPTIMIZER,'cpu',train_dl,test_dl)

  0%|          | 0/2 [00:00<?, ?it/s]

Epoch 0:
Train Loss: 0.700 | Train Acc: 0.70 | Train DSC: 0.52| Val Loss: 0.435 | Val Acc: 0.84 | Val DSC: 0.45
Epoch 1:


KeyboardInterrupt: 

In [78]:
dice_score(cams,M,num_classes=2).sum().item()

0.4823852777481079